# Dominando la Integración de Datos con Merge (Joins) en Pandas

## 🎯 Objetivos
Al finalizar este notebook, serás capaz de:
- Comprender y aplicar los cuatro tipos principales de uniones: **Inner, Left, Right y Outer**.
- Utilizar el método `merge()` para vincular DataFrames mediante claves comunes.
- Gestionar la pérdida o aparición de valores nulos (`NaN`) tras una unión.
- Usar el parámetro `indicator` para auditar el origen de los datos integrados.

## 📖 Introducción

En la mayoría de los proyectos de datos, la información se almacena de forma normalizada (dividida en tablas para evitar redundancia). El proceso de volver a unir estas tablas para el análisis se conoce como **Join** (unión).

Pandas implementa esta funcionalidad a través de `merge()`, que funciona de manera muy similar a los `JOIN` de SQL. La clave del éxito en un `merge` es elegir el tipo de unión correcto según la pregunta que queramos responder.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path(".")

# Carga de datos optimizada
df_movies = pd.read_csv(DATA_PATH / "IMDb movies.csv", low_memory=False)
df_ratings = pd.read_csv(DATA_PATH / "IMDb ratings.csv")

df_movies = df_movies[['imdb_title_id', 'title', 'year', 'genre', 'country']]
df_ratings = df_ratings[['imdb_title_id', 'total_votes', 'mean_vote']]

## 1. Inner Join (Intersección)

### 💡 Intuición
El Inner Join es el más restrictivo: **solo conserva las filas donde la clave existe en AMBOS DataFrames**. Si una película tiene datos descriptivos pero no tiene calificaciones, desaparecerá del resultado final.

### 🛠️ Diagrama de Concepto
```
  [ Tabla A ]       [ Tabla B ]
      ( Key 1 ) <---> ( Key 1 )  ---> [ RESULTADO ]
      ( Key 2 )       ( Key 3 )      ( Solo Key 1 )
```

In [ ]:
# Ejemplo con datos sintéticos para claridad
df_a = pd.DataFrame({'id': ['1', '2', '3'], 'nombre': ['Ana', 'Beto', 'Carla']})
df_b = pd.DataFrame({'id': ['2', '3', '4'], 'sueldo': [2000, 3000, 4000]})

df_inner = df_a.merge(df_b, on='id', how='inner')

print("--- Inner Join (Solo coincidencias) ---")
display(df_inner)

## 2. Left Join (Preservación de la Izquierda)

### 💡 Intuición
El Left Join conserva **todas las filas de la tabla izquierda**, sin importar si hay una coincidencia en la derecha. Si no hay coincidencia, Pandas rellena los huecos con `NaN`.

Es el más usado cuando queremos enriquecer una tabla principal con información adicional opcional.

In [ ]:
df_left = df_a.merge(df_b, on='id', how='left')

print("--- Left Join (Toda la Tabla A) ---")
display(df_left)

## 3. Right Join (Preservación de la Derecha)

### 💡 Intuición
Es la inversa del Left Join: conserva **todas las filas de la tabla derecha**. Se usa menos frecuentemente, ya que normalmente se puede lograr el mismo resultado invirtiendo el orden de los DataFrames en un Left Join.

In [ ]:
df_right = df_a.merge(df_b, on='id', how='right')

print("--- Right Join (Toda la Tabla B) ---")
display(df_right)

## 4. Outer Join (Unión Total)

### 💡 Intuición
El Outer Join es la unión más inclusiva: **conserva todas las filas de ambas tablas**. Si hay coincidencias, las une; si no, deja `NaN` en el lado que falta.

### 🛠️ Diagrama de Concepto
```
 [ Tabla A ]            [ Tabla B ]
      |                       |
      +----------+------------+
                 |
                 v
     [ RESULTADO: A ∪ B (TODO) ]
```

In [ ]:
df_outer = df_a.merge(df_b, on='id', how='outer')

print("--- Outer Join (Todo de ambos) ---")
display(df_outer)

## 5. Auditoría con el parámetro `indicator`
### 💡 Intuición
Cuando hacemos un `outer merge`, es difícil saber de dónde vino cada fila. El parámetro `indicator=True` añade una columna llamada `_merge` que nos dice si el registro estaba en `left_only`, `right_only` o en `both`.

In [ ]:
df_audit = df_a.merge(df_b, on='id', how='outer', indicator=True)

print("--- Auditoría de Unión ---")
display(df_audit)

# Ejemplo: Filtrar solo los registros que NO coinciden (discrepancias)
discrepancies = df_audit.query("_merge != 'both'")
print("\n--- Registros sin coincidencia ---")
display(discrepancies)

## 📝 Ejercicios de Práctica

**Ejercicio 1**: Realiza un `inner join` entre `df_movies` y `df_ratings`. ¿Cuántas películas quedaron en el resultado final comparado con el dataset original de películas?

**Ejercicio 2**: Utiliza un `left join` para unir las películas con sus calificaciones. Identifica cuántas películas no tienen ninguna calificación asociada (donde `mean_vote` es `NaN`).

**Ejercicio 3**: Crea dos DataFrames pequeños con claves que se solapen parcialmente. Realiza un `outer join` con `indicator=True` y filtra el resultado para mostrar solo las filas que existen exclusivamente en la segunda tabla.

In [ ]:
# Solución Ejercicio 1
# TODO: Implementar aquí
pass

In [ ]:
# Solución Ejercicio 2
# TODO: Implementar aquí
pass

## 📋 Resumen Rápido

| Tipo de Join | Lógica | Resultado |
| :--- | :--- | :--- |
| **Inner** | Intersección | Solo coincidencias en ambos |
| **Left** | Preservación Izq. | Todo de A + Coincidencias de B |
| **Right** | Preservación Der. | Todo de B + Coincidencias de A |
| **Outer** | Unión | Absolutamente todo de ambos |